# Data Cleaning & Standardization
## St. Paul Neighborhood Health Project

This notebook standardizes neighborhoods, dates, and other key fields across all datasets.

## Setup & Load Data

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print('Loading datasets...')
perms = pd.read_csv('../data/Approved_Building_Permits_-7890413957898939046.csv')
crime = pd.read_csv('../data/Crime_Incident_Report.csv')
requests = pd.read_csv('../data/Resident_Service_Requests_7024824928576740068.csv')

print(f'✓ Permits: {len(perms):,} rows')
print(f'✓ Crime: {len(crime):,} rows')
print(f'✓ Requests: {len(requests):,} rows')

## Step 1: Standardize Neighborhood Names

Create a mapping of all neighborhood name variations to standard names.

In [ ]:
# Get all unique neighborhood values from each dataset
print("PERMITS neighborhoods:")
perms_neighborhoods = sorted(perms['NEIGHBORHOOD'].unique())  # ADJUST COLUMN NAME
print(f"Count: {len(perms_neighborhoods)}")
print(perms_neighborhoods[:10])  # Show first 10

print("\nCRIME neighborhoods:")
crime_neighborhoods = sorted(crime['NEIGHBORHOOD'].unique())  # ADJUST COLUMN NAME
print(f"Count: {len(crime_neighborhoods)}")
print(crime_neighborhoods[:10])

print("\nREQUESTS neighborhoods:")
reqs_neighborhoods = sorted(requests['NEIGHBORHOOD'].unique())  # ADJUST COLUMN NAME
print(f"Count: {len(requests_neighborhoods)}")
print(requests_neighborhoods[:10])

In [ ]:
# Create neighborhood mapping dictionary
# Group variations of the same neighborhood together
neighborhood_mapping = {
    'Downtown': ['Downtown', 'Downtown St Paul', 'DT'],
    'North End': ['North End', 'North End-Riverview', 'NE'],
    'Como Park': ['Como Park', 'Como', 'CP'],
    'West Side': ['West Side', 'West Saint Paul', 'WS'],
    # ADD MORE MAPPINGS BASED ON YOUR DATA
    # Format: 'STANDARD_NAME': ['variation1', 'variation2', 'variation3']
}

# Create reverse mapping for quick lookup
reverse_mapping = {}
for standard_name, variations in neighborhood_mapping.items():
    for variation in variations:
        reverse_mapping[variation.strip().title()] = standard_name

print("Mapping created with standards:")
for standard in neighborhood_mapping.keys():
    print(f"  - {standard}")
print(f"\nTotal mappings: {len(reverse_mapping)}")
print(f"\nExample reverse mapping:")
print(reverse_mapping)

In [ ]:
# Apply standardization to permits
perms['NEIGHBORHOOD_STANDARD'] = perms['NEIGHBORHOOD'].str.strip().str.title().map(reverse_mapping)

# Check for unmapped values
unmapped_perms = perms[perms['NEIGHBORHOOD_STANDARD'].isnull()]['NEIGHBORHOOD'].unique()
if len(unmapped_perms) > 0:
    print(f"UNMAPPED Permits neighborhoods ({len(unmapped_perms)}):")
    print(unmapped_perms)
    print(f"\nThese need to be added to the mapping dictionary.")
else:
    print("✓ All permits neighborhoods mapped!")

# Apply to crime
crime['NEIGHBORHOOD_STANDARD'] = crime['NEIGHBORHOOD'].str.strip().str.title().map(reverse_mapping)
unmapped_crime = crime[crime['NEIGHBORHOOD_STANDARD'].isnull()]['NEIGHBORHOOD'].unique()
if len(unmapped_crime) > 0:
    print(f"\nUNMAPPED Crime neighborhoods ({len(unmapped_crime)}):")
    print(unmapped_crime)
else:
    print("✓ All crime neighborhoods mapped!")

# Apply to requests
requests['NEIGHBORHOOD_STANDARD'] = requests['NEIGHBORHOOD'].str.strip().str.title().map(reverse_mapping)
unmapped_reqs = requests[requests['NEIGHBORHOOD_STANDARD'].isnull()]['NEIGHBORHOOD'].unique()
if len(unmapped_reqs) > 0:
    print(f"\nUNMAPPED Request neighborhoods ({len(unmapped_reqs)}):")
    print(unmapped_reqs)
else:
    print("✓ All request neighborhoods mapped!")

print(f"\n\nStandardized neighborhoods ({len(neighborhood_mapping)}):")
for n in sorted(neighborhood_mapping.keys()):
    print(f"  - {n}")

## Step 2: Standardize Dates

In [ ]:
# Parse dates
perms['PERMIT_DATE'] = pd.to_datetime(perms['ISSUE_DATE'], errors='coerce')  # ADJUST COLUMN NAME
crime['CRIME_DATE'] = pd.to_datetime(crime['INCIDENT_DATE'], errors='coerce')  # ADJUST COLUMN NAME
requests['REQUEST_DATE'] = pd.to_datetime(requests['REQUEST_DATE'], errors='coerce')  # ADJUST COLUMN NAME

# Extract year for analysis
perms['YEAR'] = perms['PERMIT_DATE'].dt.year
crime['YEAR'] = crime['CRIME_DATE'].dt.year
requests['YEAR'] = requests['REQUEST_DATE'].dt.year

# Check date ranges
print("DATE RANGES:")
print(f"\nPermits:")
print(f"  Range: {perms['PERMIT_DATE'].min()} to {perms['PERMIT_DATE'].max()}")
print(f"  Missing: {perms['PERMIT_DATE'].isnull().sum():,} ({perms['PERMIT_DATE'].isnull().sum()/len(perms)*100:.1f}%)")
print(f"  Future dates: {(perms['PERMIT_DATE'] > pd.Timestamp.now()).sum()}")
print(f"  Years covered: {sorted(perms['YEAR'].dropna().unique())}")

print(f"\nCrime:")
print(f"  Range: {crime['CRIME_DATE'].min()} to {crime['CRIME_DATE'].max()}")
print(f"  Missing: {crime['CRIME_DATE'].isnull().sum():,} ({crime['CRIME_DATE'].isnull().sum()/len(crime)*100:.1f}%)")
print(f"  Future dates: {(crime['CRIME_DATE'] > pd.Timestamp.now()).sum()}")
print(f"  Years covered: {sorted(crime['YEAR'].dropna().unique())}")

print(f"\nRequests:")
print(f"  Range: {requests['REQUEST_DATE'].min()} to {requests['REQUEST_DATE'].max()}")
print(f"  Missing: {requests['REQUEST_DATE'].isnull().sum():,} ({requests['REQUEST_DATE'].isnull().sum()/len(requests)*100:.1f}%)")
print(f"  Future dates: {(requests['REQUEST_DATE'] > pd.Timestamp.now()).sum()}")
print(f"  Years covered: {sorted(requests['YEAR'].dropna().unique())}")

# Find overlap
min_year = max(perms['YEAR'].min(), crime['YEAR'].min(), requests['YEAR'].min())
max_year = min(perms['YEAR'].max(), crime['YEAR'].max(), requests['YEAR'].max())
print(f"\nOVERLAP PERIOD: {int(min_year)} to {int(max_year)}")
print(f"Analysis will focus on this period.")

## Step 3: Validate Key Fields

In [ ]:
# Crime data - validate offense types
print("Crime data quality:")
print(f"\nOffense column: 'OFFENSE'")
print(f"Unique values: {crime['OFFENSE'].nunique()}")
print(f"Missing: {crime['OFFENSE'].isnull().sum()}")
print(f"\nTop 10 offense types:")
print(crime['OFFENSE'].value_counts().head(10))

# Service requests - validate issue types
print("\n\nService requests data quality:")
print(f"\nIssue type column: 'ISSUE_TYPE'")
print(f"Unique values: {requests['ISSUE_TYPE'].nunique()}")
print(f"Missing: {requests['ISSUE_TYPE'].isnull().sum()}")
print(f"\nTop 10 issue types:")
print(requests['ISSUE_TYPE'].value_counts().head(10))

# Permits data - validate costs
print("\n\nPermits data quality:")
print(f"\nEstimated cost column: 'ESTIMATED_COST'")
print(f"Data type: {perms['ESTIMATED_COST'].dtype}")
print(f"Missing: {perms['ESTIMATED_COST'].isnull().sum()}")
print(f"Negative values: {(perms['ESTIMATED_COST'] < 0).sum()}")
print(f"\nSummary:")
print(perms['ESTIMATED_COST'].describe())

## Step 4: Remove Duplicates & Invalid Records

In [ ]:
# Check for duplicates
print("DUPLICATE CHECK:")
print(f"\nPermits - exact duplicates: {perms.duplicated().sum()}")
print(f"Crime - exact duplicates: {crime.duplicated().sum()}")
print(f"Requests - exact duplicates: {requests.duplicated().sum()}")

# Remove rows with missing key fields
perms_clean = perms.dropna(subset=['NEIGHBORHOOD_STANDARD', 'YEAR'])
crime_clean = crime.dropna(subset=['NEIGHBORHOOD_STANDARD', 'YEAR'])
requests_clean = requests.dropna(subset=['NEIGHBORHOOD_STANDARD', 'YEAR'])

print(f"\nAfter removing rows with missing key fields:")
print(f"Permits: {len(perms_clean):,} (removed {len(perms) - len(perms_clean):,})")
print(f"Crime: {len(crime_clean):,} (removed {len(crime) - len(crime_clean):,})")
print(f"Requests: {len(requests_clean):,} (removed {len(requests) - len(requests_clean):,})")

## Step 5: Final Data Quality Report

In [ ]:
print("FINAL DATA QUALITY REPORT")
print("="*80)

print(f"\nPERMITS:")
print(f"  Records: {len(perms_clean):,}")
print(f"  Neighborhoods: {perms_clean['NEIGHBORHOOD_STANDARD'].nunique()}")
print(f"  Years: {sorted(perms_clean['YEAR'].dropna().unique())}")
print(f"  Missing neighborhoods: {perms_clean['NEIGHBORHOOD_STANDARD'].isnull().sum()}")

print(f"\nCRIME:")
print(f"  Records: {len(crime_clean):,}")
print(f"  Neighborhoods: {crime_clean['NEIGHBORHOOD_STANDARD'].nunique()}")
print(f"  Years: {sorted(crime_clean['YEAR'].dropna().unique())}")
print(f"  Missing neighborhoods: {crime_clean['NEIGHBORHOOD_STANDARD'].isnull().sum()}")

print(f"\nREQUESTS:")
print(f"  Records: {len(requests_clean):,}")
print(f"  Neighborhoods: {requests_clean['NEIGHBORHOOD_STANDARD'].nunique()}")
print(f"  Years: {sorted(requests_clean['YEAR'].dropna().unique())}")
print(f"  Missing neighborhoods: {requests_clean['NEIGHBORHOOD_STANDARD'].isnull().sum()}")

print(f"\nNEIGHBORHOODS REPRESENTED IN ALL DATASETS:")
perms_hoods = set(perms_clean['NEIGHBORHOOD_STANDARD'].unique())
crime_hoods = set(crime_clean['NEIGHBORHOOD_STANDARD'].unique())
reqs_hoods = set(requests_clean['NEIGHBORHOOD_STANDARD'].unique())

common_hoods = perms_hoods & crime_hoods & reqs_hoods
print(f"  Common to all 3: {len(common_hoods)} neighborhoods")
print(f"  List: {sorted(common_hoods)}")

print(f"\nNEIGHBORHOODS ONLY IN SOME DATASETS:")
perms_only = perms_hoods - crime_hoods - reqs_hoods
crime_only = crime_hoods - perms_hoods - reqs_hoods
reqs_only = reqs_hoods - perms_hoods - crime_hoods
if perms_only:
    print(f"  Only in Permits: {perms_only}")
if crime_only:
    print(f"  Only in Crime: {crime_only}")
if reqs_only:
    print(f"  Only in Requests: {reqs_only}")

print(f"\n✓ Data cleaning complete!")
print(f"Ready for aggregation and indicator development.")